# Experimento — Correção Automatizada com e sem RAG (Livro de História)

Este notebook executa o experimento completo via API do backend, usando como fonte o livro  
**História — Ensino Médio, 2ª Ed. (SEED-PR, 2007)**.

**Fluxo:**
1. Setup (registro, login, criação de turma e alunos)
2. upload do PDF de História + publish

## 0. Configuração

In [1]:
import requests
import json
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

BASE_URL = "http://localhost:8000"
PDF_PATH = Path("../notebook/historia.pdf")
OUTPUT_DIR = Path("../notebook/results")
OUTPUT_DIR.mkdir(exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

assert PDF_PATH.exists(), f"PDF não encontrado em {PDF_PATH.resolve()}"

# Estado global do experimento
state = {}

def api(method, path, **kwargs):
    """Helper para chamadas à API com auth automático."""
    headers = kwargs.pop("headers", {})
    if "token" in state:
        headers["Authorization"] = f"Bearer {state['token']}"
    resp = getattr(requests, method)(f"{BASE_URL}{path}", headers=headers, **kwargs)
    if not resp.ok:
        print(f"❌ {method.upper()} {path} → {resp.status_code}")
        print(resp.text[:500])
    return resp

def save_csv(df, name):
    """Salva DataFrame como CSV em OUTPUT_DIR."""
    path = OUTPUT_DIR / f"{name}_{TIMESTAMP}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  💾 {path}")
    return path

print("✅ Configuração carregada")
print(f"   Backend: {BASE_URL}")
print(f"   PDF:     {PDF_PATH.resolve()}")
print(f"   Output:  {OUTPUT_DIR.resolve()}")
print(f"   Run ID:  {TIMESTAMP}")

✅ Configuração carregada
   Backend: http://localhost:8000
   PDF:     C:\Users\Maycon Mendes\OneDrive\Área de Trabalho\IFF\ai-grading-system\notebook\historia.pdf
   Output:  C:\Users\Maycon Mendes\OneDrive\Área de Trabalho\IFF\ai-grading-system\notebook\results
   Run ID:  20260407_004113


## 1. Dados do Experimento

Questões extraídas das 4 Unidades Temáticas do livro, critérios e respostas simuladas em 4 níveis de qualidade.

In [2]:
# ── Questões (baseadas nas Unidades Temáticas do livro) ───────────────────────
# Unidade I  — Trabalho escravo e trabalho livre (Folhas 1-5)
# Unidade II — Urbanização e industrialização     (Folhas 6-11)
# Unidade III— O Estado e as relações de poder    (Folhas 12-15)
# Unidade IV — Movimentos sociais, políticos e culturais (Folhas 16-20)

QUESTIONS = [
    {
        "id": "Q1",
        "order": 1,
        "points": 10.0,
        "statement": (
            "(Unidade I — Trabalho escravo e trabalho livre) "
            "Explique a transição do trabalho escravo para o trabalho assalariado no Brasil "
            "e nos Estados Unidos durante o século XIX. Aponte as principais semelhanças e "
            "diferenças entre os dois processos, considerando o contexto de consolidação do "
            "capitalismo em cada sociedade."
        ),
    },
    {
        "id": "Q2",
        "order": 2,
        "points": 10.0,
        "statement": (
            "(Unidade II — Urbanização e industrialização) "
            "Analise o processo de urbanização e industrialização no Brasil comparando-o "
            "com o ocorrido nos países europeus no século XIX. Discuta os impactos sociais, "
            "econômicos e culturais desse processo, destacando o papel do Porto de Paranaguá "
            "na expansão do capitalismo no Paraná."
        ),
    },
    {
        "id": "Q3",
        "order": 3,
        "points": 10.0,
        "statement": (
            "(Unidade III — O Estado e as relações de poder) "
            "Discuta como o conceito de Estado evoluiu desde o mundo antigo e medieval até "
            "a formação dos Estados nacionais modernos. Explique de que forma as relações de "
            "poder e a violência estatal se manifestam na análise de Michel Foucault e dos "
            "historiadores da Nova Esquerda Inglesa."
        ),
    },
]

# ── Critérios de correção ─────────────────────────────────────────────────────
# Pesos: Correção histórica 35%, Completude 25%, Uso de conceitos/terminologia 20%,
#        Evidências/fontes 10%, Análise crítica 10%.
# max_points = weight * 10.0 (pontos totais da questão)

CRITERIAS = [
    {
        "name": "Correção histórica",
        "description": (
            "Avalia se os fatos, datas, processos e interpretações históricas apresentados "
            "estão corretos, sem erros factuais ou anacronismos."
        ),
        "max_points": 3.5,
        "weight": 0.35,
    },
    {
        "name": "Completude da resposta",
        "description": (
            "Avalia se todos os aspectos solicitados no enunciado foram abordados "
            "de forma adequada, sem omitir dimensões relevantes da questão."
        ),
        "max_points": 2.5,
        "weight": 0.25,
    },
    {
        "name": "Uso de conceitos e terminologia",
        "description": (
            "Avalia o uso correto dos conceitos historiográficos (relações de trabalho, "
            "de poder e culturais) e da terminologia específica da disciplina de História, "
            "conforme trabalhados no livro didático."
        ),
        "max_points": 2.0,
        "weight": 0.20,
    },
    {
        "name": "Uso de evidências e fontes",
        "description": (
            "Avalia se a resposta apresenta exemplos concretos, referências a documentos, "
            "autores ou fontes históricas que fundamentem as afirmações realizadas."
        ),
        "max_points": 1.0,
        "weight": 0.10,
    },
    {
        "name": "Análise crítica e profundidade",
        "description": (
            "Avalia se a resposta vai além da descrição dos fatos, demonstrando capacidade "
            "de interpretar, problematizar e relacionar os conteúdos históricos com o "
            "contexto contemporâneo."
        ),
        "max_points": 1.0,
        "weight": 0.10,
    },
]

# ── Respostas simuladas em 4 níveis de qualidade ──────────────────────────────
# Nível A: excelente | B: bom | C: regular | D: fraco

ANSWERS = {
    "Q1": {
        "A": (
            "A transição do trabalho escravo para o trabalho assalariado ocorreu de forma "
            "distinta no Brasil e nos EUA, embora ambos os processos estejam inseridos na "
            "expansão capitalista do século XIX. Nos EUA, a abolição foi resultado da "
            "Guerra Civil (1861-1865), que opôs o Norte industrializado ao Sul escravista. "
            "No Brasil, a abolição foi gradual — Lei do Ventre Livre (1871), Lei dos "
            "Sexagenários (1885) e Lei Áurea (1888) — impulsionada por pressões "
            "econômicas e movimentos abolicionistas. Em ambos os países, a mão de obra "
            "liberta encontrou dificuldades de inserção no mercado de trabalho livre, "
            "enfrentando discriminação e condições análogas à escravidão. A principal "
            "diferença está no ritmo: enquanto os EUA romperam abruptamente, o Brasil "
            "adotou uma transição lenta que favoreceu a substituição pelo trabalhador "
            "imigrante europeu, especialmente nas lavouras de café em São Paulo. "
            "Hobsbawm e Thompson alertam que a formação do trabalho assalariado não "
            "representa a superação das relações de exploração, mas sua reconfiguração "
            "dentro da lógica capitalista."
        ),
        "B": (
            "No Brasil, a escravidão foi abolida em 1888 com a Lei Áurea, após um longo "
            "processo que incluiu leis gradualistas. Nos EUA, a abolição veio com o fim "
            "da Guerra Civil em 1865. Nos dois países, os ex-escravizados passaram a "
            "trabalhar de forma assalariada, mas enfrentaram muitas dificuldades. A "
            "principal semelhança é que em ambos os casos os trabalhadores libertos "
            "ficaram marginalizados socialmente. A diferença é que no Brasil houve uma "
            "substituição pelo imigrante europeu, enquanto nos EUA os ex-escravizados "
            "continuaram trabalhando nas fazendas do sul sob outras formas de exploração. "
            "O capitalismo se consolidou aproveitando essa mão de obra mais barata."
        ),
        "C": (
            "A escravidão acabou no Brasil com a Lei Áurea em 1888. Nos EUA acabou depois "
            "da guerra entre o Norte e o Sul. Depois disso os escravos passaram a receber "
            "salários pelo trabalho. No Brasil vieram muitos imigrantes para trabalhar. "
            "Os dois países tiveram dificuldades na transição mas conseguiram desenvolver "
            "a economia capitalista com trabalho livre."
        ),
        "D": (
            "A escravidão acabou porque os países decidiram que era errado escravizar "
            "pessoas. Depois disso todo mundo passou a trabalhar com salário e a economia "
            "melhorou. No Brasil a abolição foi em 1888 e nos EUA foi antes."
        ),
    },
    "Q2": {
        "A": (
            "A urbanização e industrialização europeia do século XIX foi impulsionada pela "
            "Revolução Industrial, gerando intenso êxodo rural e formação de um proletariado "
            "urbano. No Brasil, esse processo ocorreu de forma tardia e dependente, a partir "
            "do final do século XIX e início do XX, associado ao capital cafeeiro e à "
            "imigração europeia. Os impactos sociais incluíram o surgimento de favelas, "
            "condições precárias de trabalho e movimentos operários. No Paraná, o Porto de "
            "Paranaguá foi peça central na integração da economia regional ao mercado "
            "internacional, escoando erva-mate e madeira. Seu desenvolvimento refletiu "
            "diretamente a expansão do capitalismo, atraindo investimentos em ferrovias "
            "(como a Estrada de Ferro Paranaguá-Curitiba) e gerando relações de trabalho "
            "específicas no porto. A comparação com a Europa revela que, enquanto lá a "
            "industrialização foi motor da urbanização, no Brasil o processo foi invertido: "
            "a urbanização precedeu a industrialização plena."
        ),
        "B": (
            "Na Europa, a industrialização do século XIX provocou grande crescimento das "
            "cidades e surgimento da classe operária. No Brasil, o processo foi mais lento "
            "e começou com a chegada de imigrantes e o crescimento do café. O Porto de "
            "Paranaguá foi importante para exportar produtos do Paraná como erva-mate e "
            "madeira, integrando a região à economia capitalista. Os impactos sociais "
            "incluíram a formação de bairros operários e conflitos trabalhistas. A grande "
            "diferença é que a Europa industrializou primeiro e depois urbanizou, enquanto "
            "o Brasil teve um caminho diferente."
        ),
        "C": (
            "A industrialização na Europa aconteceu por causa da Revolução Industrial. No "
            "Brasil foi mais tarde. O Porto de Paranaguá ajudou o Paraná a se desenvolver "
            "economicamente. Com a industrialização as cidades cresceram muito e surgiram "
            "problemas sociais como pobreza e falta de moradia."
        ),
        "D": (
            "A industrialização fez as cidades crescerem. O Porto de Paranaguá é importante "
            "para o Paraná. No Brasil a industrialização aconteceu mais devagar do que na Europa."
        ),
    },
    "Q3": {
        "A": (
            "O Estado, nas sociedades antigas e medievais, organizava-se em torno de "
            "estruturas como a polis grega, o Império Romano e os reinos feudais europeus, "
            "nos quais o poder se legitimava por laços religiosos, militares e de sangue. "
            "Com a modernidade, emergem os Estados nacionais baseados na soberania "
            "territorial e no monopólio legítimo da violência, conforme Max Weber. A Nova "
            "Esquerda Inglesa, representada por Hobsbawm e Thompson, critica a visão "
            "político-estatal tradicional, propondo analisar o poder a partir das condições "
            "materiais e das classes sociais. Já Michel Foucault radicaliza essa perspectiva "
            "ao propor o conceito de 'micropoderes': o poder não reside apenas no Estado, "
            "mas se exerce em instituições como escolas, prisões, hospitais e famílias, "
            "permeando todas as relações sociais. Para Foucault, o Estado moderno utiliza "
            "técnicas de controle e normalização dos corpos (biopoder). Ambas as "
            "perspectivas contribuem para uma análise crítica das relações de dominação "
            "e resistência ao longo da história."
        ),
        "B": (
            "Nas sociedades antigas, como a Grécia e Roma, o Estado era organizado de "
            "forma diferente do Estado moderno. Na Idade Média, o poder estava nas mãos "
            "dos reis e da Igreja. Com o tempo, surgiram os Estados nacionais com "
            "fronteiras definidas e leis próprias. Foucault mostrou que o poder não está "
            "só no Estado, mas também nas instituições do cotidiano. A Nova Esquerda "
            "Inglesa preferia analisar o poder a partir das classes sociais e das relações "
            "econômicas. Assim, o estudo do Estado precisa considerar tanto a perspectiva "
            "política quanto a social e cultural."
        ),
        "C": (
            "O Estado antigamente era muito diferente. Na Grécia existia a democracia e "
            "em Roma havia o Império. Na Idade Média o poder era dos reis. Depois vieram "
            "os países modernos. Foucault falou sobre como o poder age nas instituições. "
            "O Estado moderno tem leis e fronteiras."
        ),
        "D": (
            "O Estado mudou muito ao longo da história. Na Antiguidade era diferente de hoje. "
            "Hoje o Estado tem presidente e leis. Foucault estudou o poder."
        ),
    },
}

# ── Alunos simulados ──────────────────────────────────────────────────────────
STUDENTS = [
    {"name": "Ana Nível A",   "email": "ana_a@escola.edu",   "level": "A"},
    {"name": "Bruno Nível B", "email": "bruno_b@escola.edu", "level": "B"},
    {"name": "Carla Nível C", "email": "carla_c@escola.edu", "level": "C"},
    {"name": "Diego Nível D", "email": "diego_d@escola.edu", "level": "D"},
]

print(f"✅ Dados carregados: {len(QUESTIONS)} questões, {len(CRITERIAS)} critérios, "
      f"{len(STUDENTS)} alunos, {len(QUESTIONS)*len(STUDENTS)} respostas")

✅ Dados carregados: 3 questões, 5 critérios, 4 alunos, 12 respostas


## 2. Setup — Registro, Login, Turma e Alunos

In [3]:
import uuid

RUN_SUFFIX = TIMESTAMP[-6:]  # últimos 6 dígitos para unicidade

# Registro do professor
teacher_email = f"prof.historia.{RUN_SUFFIX}@escola.edu"
teacher_password = "Historia@2024"

r = api("post", "/auth/register", json={
    "name": "Prof. História Experimento",
    "email": teacher_email,
    "password": teacher_password,
})
assert r.ok, "Falha no registro"
print(f"✅ Professor registrado: {teacher_email}")

# Login
r = api("post", "/auth/login", data={"username": teacher_email, "password": teacher_password})
assert r.ok, "Falha no login"
state["token"] = r.json()["access_token"]
print("✅ Login realizado")

# Criação da turma
r = api("post", "/classes", json={"name": f"Historia-{RUN_SUFFIX}"})
assert r.ok, "Falha ao criar turma"
state["class_id"] = r.json()["id"]
print(f"✅ Turma criada: {state['class_id']}")

# Criação dos alunos
state["students"] = []
for s in STUDENTS:
    r = api("post", "/students", json={
        "name": s["name"],
        "email": f"{RUN_SUFFIX}.{s['email']}",
        "class_id": state["class_id"],
    })
    assert r.ok, f"Falha ao criar aluno {s['name']}"
    state["students"].append({**s, "id": r.json()["id"]})
    print(f"  👤 {s['name']} criado")

print(f"\n✅ Setup concluído — {len(state['students'])} alunos registrados")

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /auth/register (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] Nenhuma conexão pôde ser feita porque a máquina de destino as recusou ativamente"))

## 3. Upload do PDF de História + publish

In [ ]:
def run_condition(label, upload_pdf=True):
    """Executa uma condição completa (com ou sem RAG) e retorna os resultados."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")

    results = []

    # Criação do exame
    r = api("post", "/exams", json={
        "title": f"Historia {label} {RUN_SUFFIX}",
        "class_id": state["class_id"],
        "due_date": (datetime.now() + timedelta(days=7)).isoformat(),
    })
    assert r.ok, "Falha ao criar exame"
    exam_id = r.json()["id"]
    print(f"  📝 Exame criado: {exam_id}")

    # Upload do PDF (apenas na Condição A)
    if upload_pdf:
        with open(PDF_PATH, "rb") as f:
            r = api("post", f"/exams/{exam_id}/document",
                    files={"file": (PDF_PATH.name, f, "application/pdf")})
        assert r.ok, "Falha no upload do PDF"
        print(f"  📄 PDF enviado: {PDF_PATH.name} ({PDF_PATH.stat().st_size/1024/1024:.1f} MB)")
        time.sleep(5)  # aguarda processamento

    # Criação das questões e critérios
    question_ids = {}
    for q in QUESTIONS:
        r = api("post", f"/exams/{exam_id}/questions", json={
            "order": q["order"],
            "statement": q["statement"],
            "points": q["points"],
        })
        assert r.ok, f"Falha ao criar questão {q['id']}"
        q_id = r.json()["id"]
        question_ids[q["id"]] = q_id

        for c in CRITERIAS:
            r = api("post", f"/questions/{q_id}/criterias", json={
                "name": c["name"],
                "description": c["description"],
                "max_points": c["max_points"],
                "weight": c["weight"],
            })
            assert r.ok, f"Falha ao criar critério {c['name']}"

    print(f"  ✅ {len(QUESTIONS)} questões e {len(CRITERIAS)} critérios criados")

    # Publicação do exame
    r = api("post", f"/exams/{exam_id}/publish")
    assert r.ok, "Falha ao publicar exame"
    print(f"  🚀 Exame publicado")

    # Envio das respostas e coleta das correções
    for student in state["students"]:
        level = student["level"]
        s_id = student["id"]

        # Criação da submissão
        r = api("post", f"/exams/{exam_id}/submissions", json={"student_id": s_id})
        assert r.ok, f"Falha ao criar submissão para {student['name']}"
        sub_id = r.json()["id"]

        # Envio das respostas por questão
        for q in QUESTIONS:
            r = api("post", f"/submissions/{sub_id}/answers", json={
                "question_id": question_ids[q["id"]],
                "text": ANSWERS[q["id"]][level],
            })
            assert r.ok, f"Falha ao enviar resposta {q['id']} para {student['name']}"

        # Solicitar correção automática
        r = api("post", f"/submissions/{sub_id}/review")
        assert r.ok, f"Falha ao solicitar revisão para {student['name']}"
        raw_review = r.json()

        # Coleta das notas por questão
        for q in QUESTIONS:
            q_id = question_ids[q["id"]]
            score = None
            feedback = ""
            for item in raw_review.get("reviews", []):
                if item.get("question_id") == q_id:
                    score = item.get("score")
                    feedback = item.get("feedback", "")
                    break

            results.append({
                "condition": label,
                "student": student["name"],
                "level": level,
                "question": q["id"],
                "score": score,
                "max_score": q["points"],
                "feedback_length": len(feedback),
                "exam_id": exam_id,
                "submission_id": sub_id,
            })

        print(f"  ✅ {student['name']} (nível {level}) — revisão coletada")

    df = pd.DataFrame(results)
    print(f"\n  Média geral ({label}): {df['score'].mean():.2f}")
    return {"df": df, "exam_id": exam_id}


cond_a = run_condition("Condição A — Com RAG", upload_pdf=True)